# Kuroshio R5.9 — 冻结强 LSTM / R53 混合主干 + 跨双分支 EOF 动力学残差校正

**目的**：针对 R5.7 的「30 天不足、不同年份不稳定」，不再扩增失败的空间/时间卷积。保留双分支 train-only EOF 路径、180 天历史、原 LSTM / R53 混合模型的全部预报输出，额外训练一个 **低秩、近零初始化**的跨分支物理残差校正模块。与**参数更多**的容量匹配 LSTM 直接比较，只有内层验证优于父模型才允许使用校正。**没有测试集胜出保证；失败按事实报告。**

**合同**：原冻结数据不可改；父模型与 R5.7 完全相同的 train/val/QA 拆分、EOF、权重、随机种子和训练配方；所有内层选择只看 QA 之前的数据。研发用 2008/2016 QA，若通过再在 2010/2018 年追加确认。旧 QA 均曾被查看，不能称为全盲独立测试。结果自动 ZIP + SHA256 + 可点击下载链接。默认不进入正式多年训练。

**机制消融**：`R59-OwnBranch`（仅用对应分支的历史状态）与 `R59-CrossBranch`（同时考虑台湾段和日本黑潮延伸段的状态变化），共享相同强 LSTM 父模型。对比 `Matched-LSTM` 与参数更大的 `R53-CapacityMatchedLSTM`，不篡改评价真值。

**新增核心候选**：`R59-HybridCoupled`，以原 R53-ChannelConv-PCConcat（保留空间 TCN/EOF 旁路/LSTM）为父模型，然后用同一低秩校正器学习跨分支偏差；因此真正测试的是原混合方法的增强，而非仅在纯 LSTM 上后处理。

In [ ]:
# ===== 0. 配置：先核实三个目录；改好后从头顺序运行 =====
from dataclasses import dataclass, asdict
from pathlib import Path
import os, json, math, gc, time, random, hashlib, warnings, sys
import numpy as np
import pandas as pd

@dataclass
class CFG:
    ROOT: str = '/mnt/d/KuroshioDaily1993_2025'
    DATASET_REL: str = 'final/dataset/daily_dual_branch_paths_1993_2025_FINAL.npz'
    HANDOFF_REL: str = 'final/reports/LONG_DAILY_1993_2025_HANDOFF.json'
    QC_INDEX_REL: str = 'final/reports/最终逐日路径索引.csv'
    MANUAL_QUEUE_REL: str = 'reports/07_人工复核队列.csv'
    MANUAL_METADATA_REL: str = 'manual_qc/metadata'
    OUTPUT_REL: str = 'R5_9_跨分支残差校正资格实验'
    HISTORY: int = 180
    LEADS: tuple = (30, 60, 90)
    TEST_YEARS: tuple = (2005, 2009, 2013, 2017, 2021, 2024)
    PURGE_DAYS: int = 7
    MAX_INPUT_GAP: int = 2
    MAX_MISSING_FRAC: float = 0.03
    MIN_DATA_YEARS: int = 25
    MIN_ANNUAL_COVERAGE: float = 0.85
    MIN_TRAIN: int = 500
    MIN_VAL: int = 100
    MIN_TEST: int = 100
    DROP_BRANCH1_FIRST: int = 12  # 43->31; same old representation
    BRANCH2_MAX_LON: float = 145.0
    EOF_VARIANCE_TARGET: float = 0.99
    BATCH_SIZE: int = 32
    EPOCHS_B1: int = 70
    EPOCHS_B2: int = 70
    PATIENCE: int = 12
    LR: float = 0.001
    WEIGHT_DECAY: float = 0.0001
    DROPOUT: float = 0.30
    SEEDS: tuple = (20,21,22,23,24)
    AR_LAGS: tuple = (3, 7)
    AR_ALPHAS: tuple = (0.1, 1., 10.)
    CLIM_HALF_WINDOW_DAYS: int = 15
    CLIM_MIN_SAMPLES: int = 10
    BOOT_N: int = 1000
    BOOT_BLOCK_DAYS: int = 30
    FORCE_TRAIN: bool = False
    USE_AMP: bool = True
    RUN_FULL_TRAINING: bool = False   # R5.3仅机制资格实验，严禁直接转长训
    TRAIN_OTHER_HYBRIDS: bool = False # 下一轮网络结构比较可启用，不改baseline主表
    FIG_DPI: int = 300
    PILOT_TEST_YEARS: tuple = (2009, 2017)  # QA: 2008/2016, inner early-stop: 2007/2015
    PILOT_SEEDS: tuple = (31, 32, 33)
    PILOT_EPOCHS: int = 24
    PILOT_PATIENCE: int = 7
    GATE_MIN_GAIN_KM: float = 0.50
    GATE_MAX_LEAD_REGRESSION_KM: float = 2.0

    CONFIRM_PILOT_YEARS: tuple = (2013, 2021)  # QA 2012/2020; 曾经被研发流程涉及，不能称独立盲测
    MAX_PARAM_OVERHEAD_RATIO: float = 0.15
    REQUIRE_ALL_LEADS_NONINFERIOR: bool = True
    REQUIRE_BOTH_QA_YEARS_GAIN: bool = True
    ARCHIVE_INCLUDE_WEIGHTS: bool = False
    ARCHIVE_INCLUDE_CACHE: bool = False
    ARCHIVE_MAX_FILE_MB: int = 150
    RUN_CONFIRMATION: bool = True   # 仅阶段一通过才运行阶段二；不自动正式长训
    RAW_TRACK_DIR_REL: str = 'final/tracks_final'
    MAX_MISSING_DP_DAY_FRAC: float = 0.005
    MAX_DP_TEMPORAL_P95_RATIO: float = 1.40
    RAW_EVAL_ISSUES_PER_QA_YEAR: int = 36
    REBUILD_DP: bool = False
    DP_REPORT_EVERY: int = 250
    ARCHIVE_INCLUDE_WEIGHTS: bool = False
    ARCHIVE_INCLUDE_CACHE: bool = False

cfg=CFG(); ROOT=Path(cfg.ROOT); DS=ROOT/cfg.DATASET_REL; OUT=ROOT/cfg.OUTPUT_REL
DATA=OUT/'实验数据'; CKPT=OUT/'模型权重'; PRED=OUT/'逐样本预测'; REP=OUT/'报告'; FZH=OUT/'图表'/'中文'; FEN=OUT/'图表'/'英文'
for d in (DATA,CKPT,PRED,REP,FZH,FEN): d.mkdir(parents=True,exist_ok=True)
print(json.dumps(asdict(cfg),indent=2,ensure_ascii=False))
ACTIVE_REP='Original'
RAW_TRACK_DIR=ROOT/cfg.RAW_TRACK_DIR_REL
if not RAW_TRACK_DIR.is_dir(): raise FileNotFoundError(f'找不到原始完整流轴目录: {RAW_TRACK_DIR}')
DERIVED=OUT/'派生表示_不覆盖冻结数据';DERIVED.mkdir(exist_ok=True,parents=True)
ARCHIVE_PATH=ROOT/'R59_上传给ChatGPT_跨分支残差优化资格实验.zip'
ARCHIVE_SHA=Path(str(ARCHIVE_PATH)+'.sha256')
if not DS.exists(): raise FileNotFoundError(f'未发现真正的冻结数据：{DS}。请先运行人工QC notebook最后的冻结与数据构建单元。')
# R5.7 不启用 DP，不改变冻结目标。训练合同使用资产内容 SHA256。
ACTIVE_REP='Original'
A2_ORIGINAL=None
sign=hashlib.sha256(DS.read_bytes()).hexdigest()
PARENT_R57=ROOT/'R5_7_原表示混合模型稳定性资格实验'/'模型权重'
print('R5.7父模型权重目录:',PARENT_R57)


In [ ]:
# ===== 1. GPU与字体/依赖检查（不合格立即停止，避免白训练）=====
import importlib.util
required=('torch','scipy','sklearn','matplotlib','tqdm','pandas','numpy')
missing=[v for v in required if importlib.util.find_spec(v) is None]
if missing: raise RuntimeError('WSL环境缺依赖：'+', '.join(missing)+'；用当前Kernel对应python执行 python -m pip install torch scipy scikit-learn matplotlib tqdm pandas numpy')
import torch, torch.nn as nn
from torch.utils.data import Dataset,DataLoader
from scipy.spatial import cKDTree
from sklearn.linear_model import Ridge
import matplotlib, matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.font_manager import FontProperties
from tqdm.auto import tqdm
print('kernel=',sys.executable,'torch=',torch.__version__,'CUDA=',torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('本轮RTX5080正式训练要求CUDA GPU；请检查WSL nvidia-smi 与 PyTorch CUDA构建。')
print('GPU=',torch.cuda.get_device_name(0)); DEV=torch.device('cuda')
torch.set_float32_matmul_precision('high')
font_paths=['/mnt/c/Windows/Fonts/msyh.ttc','/mnt/c/Windows/Fonts/simhei.ttf','/mnt/c/Windows/Fonts/simsun.ttc','/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc']
CJK_NAME=None
for path in font_paths:
    if Path(path).exists():
        try:
            font_manager.fontManager.addfont(path)
            CJK_NAME=FontProperties(fname=path).get_name(); break
        except Exception: pass
if CJK_NAME is None:
    raise RuntimeError('中文字体缺失：请在WSL安装 sudo apt install fonts-noto-cjk 或检查 Windows /mnt/c/Windows/Fonts，避免绘图乱码。')
print('Chinese font=',CJK_NAME)


In [ ]:
# ===== 2. 最终数据读取、年度覆盖、人工QC冻结核对 =====
with np.load(DS,allow_pickle=False) as z:
    dates=pd.DatetimeIndex(pd.to_datetime(z['dates'])).normalize()
    b1=np.asarray(z['branch1_lon'],np.float32)
    lat1=np.asarray(z['branch1_target_lats'],float)
    b2=np.asarray(z['branch2_lat'],np.float32)
    lon2=np.asarray(z['branch2_target_lons'],float)
    source=np.asarray(z['source']).astype(str) if 'source' in z.files else np.array(['unknown']*len(dates))
if not (len(dates)==len(b1)==len(b2)==len(source)): raise RuntimeError('dates / 两分支 / source 数量不一致')
if dates.has_duplicates: raise RuntimeError('发现重复日期，禁止将其直接用于训练')
ord=np.argsort(dates.values); dates=dates[ord]; b1=b1[ord];b2=b2[ord];source=source[ord]
if not np.isfinite(b1).all() or not np.isfinite(b2).all(): raise RuntimeError('冻结数据中含 NaN/Inf')

# 物理范围异常只作质控警示（不随意删除或修改路径），便于发现断面插值外推造成的不可信日
suspicious=(~((b1>=119)&(b1<=140)).all(axis=1)) | (~((b2>=15)&(b2<=50)).all(axis=1))
if suspicious.any():
    suspect_df=pd.DataFrame({'异常日期':dates[suspicious], '原因':'双分支坐标超出宽松物理范围，需人工核对'})
    suspect_df.to_csv(DATA/'01_待人工复查的物理范围异常路径.csv',index=False,encoding='utf-8-sig')
    warnings.warn(f'发现 {suspicious.sum()} 天路径越过宽松物理边界，建议正式训练前核实此文件')

if b1.shape[1]<=cfg.DROP_BRANCH1_FIRST: raise RuntimeError('分支1维数过小')
b1=b1[:,cfg.DROP_BRANCH1_FIRST:];lat1=lat1[cfg.DROP_BRANCH1_FIRST:]
keep2=lon2<=cfg.BRANCH2_MAX_LON+1e-8;b2=b2[:,keep2];lon2=lon2[keep2]
if len(b1)!=len(b2) or len(lat1)!=b1.shape[1] or len(lon2)!=b2.shape[1]: raise RuntimeError('双分支物理坐标不匹配')
if dates.min().year>1993 or dates.max().year<2024: raise RuntimeError('数据不覆盖1993–2024所需区间，不可宣称多年全历史验证')
cal=pd.date_range(dates.min(),dates.max(),freq='D'); pos=cal.get_indexer(dates)
valid=np.zeros(len(cal),bool);valid[pos]=True
A1=np.full((len(cal),b1.shape[1]),np.nan,np.float32);A2=np.full((len(cal),b2.shape[1]),np.nan,np.float32)
A1[pos]=b1; A2[pos]=b2
# causal forward fill (ONLY history, never target). All targets later read A1/A2 raw by exact date.
def causal_fill(raw,limit): return pd.DataFrame(raw).ffill(limit=limit).to_numpy(dtype=np.float32)
F1=causal_fill(A1,cfg.MAX_INPUT_GAP);F2=causal_fill(A2,cfg.MAX_INPUT_GAP)
annual=pd.DataFrame({'日期':cal,'有效':valid});annual['年份']=annual['日期'].dt.year
cover=annual.groupby('年份',as_index=False).agg(理论天数=('日期','size'),有效天数=('有效','sum'))
cover['有效比例']=cover['有效天数']/cover['理论天数'];cover.to_csv(DATA/'01_逐年覆盖率.csv',index=False,encoding='utf-8-sig')
if (cover['有效比例']>=cfg.MIN_ANNUAL_COVERAGE).sum()<cfg.MIN_DATA_YEARS:
    raise RuntimeError('具备≥85%有效覆盖的年份不足25年，请优先排查未完成人工修复或缺失年；详情见01_逐年覆盖率.csv')
hand=ROOT/cfg.HANDOFF_REL
if hand.exists():
    ho=json.loads(hand.read_text(encoding='utf-8'))
    print('HANDOFF final_daily_paths=',ho.get('final_daily_paths'))
    if int(ho.get('final_daily_paths',-1))!=len(dates): raise RuntimeError('HANDOFF与冻结npz样本数不一致')
else: warnings.warn('未发现HANDOFF.json；冻结数据可供资格检查，但交接记录尚未核实')
idxp=ROOT/cfg.QC_INDEX_REL
if idxp.exists():
    idx=pd.read_csv(idxp)
    if 'status' in idx.columns:
        unresolved=idx[~idx['status'].isin(['final','missing_raw','unusable'])]
        if len(unresolved): raise RuntimeError(f'最终逐日索引仍有 {len(unresolved)} 条未处理状态，停止训练')
queue=ROOT/cfg.MANUAL_QUEUE_REL; mdir=ROOT/cfg.MANUAL_METADATA_REL
if queue.exists():
    qq=pd.read_csv(queue)
    qdates=pd.to_datetime(qq['date']).dt.strftime('%Y%m%d') if len(qq) else []
    missing_meta=[q for q in qdates if not (mdir/f'{q}.json').exists()]
    if missing_meta: raise RuntimeError(f'{len(missing_meta)} 个manual复核队列日期尚无处理metadata: {missing_meta[:10]}')
    bad_meta=[]
    allowed={'manual_corrected','accepted_auto_after_manual_qc','unusable'}
    for q in qdates:
        try:
            state=json.loads((mdir/f'{q}.json').read_text(encoding='utf-8')).get('status')
            if state not in allowed:bad_meta.append((q,state))
        except Exception as e:bad_meta.append((q,repr(e)))
    if bad_meta:raise RuntimeError(f'{len(bad_meta)} 条人工QC仍不是已确认状态: {bad_meta[:10]}')
# CMEMS来源切换边界连续性检查：仅警告，不把真实大蛇行直接当成数据损坏
edge=pd.Timestamp('2025-01-01')
prev=np.flatnonzero(dates<edge)
post=np.flatnonzero(dates>=edge)
if len(prev) and len(post):
    i,j=int(prev[-1]),int(post[0])
    if (dates[j]-dates[i]).days<=5:
        d1=float(np.sqrt(np.mean((b1[j]-b1[i])**2)))
        d2=float(np.sqrt(np.mean((b2[j]-b2[i])**2)))
        pd.DataFrame([{'产品交界前':dates[i],'交界后':dates[j],
                       '分支1逐日经度RMSE_度':d1,'分支2逐日纬度RMSE_度':d2}]).to_csv(
            DATA/'01_2024到2025产品交界连续性.csv',index=False,encoding='utf-8-sig')
        if d1>1.0 or d2>1.5:
            warnings.warn('2024-2025 CMEMS来源交界发生较大路径跳变，应先检查跨产品物理一致性与流轴QC')
print('DATA PASS:',len(dates),'days;',dates.min().date(),'to',dates.max().date(),'; B1',b1.shape[1],'; B2',b2.shape[1])
display(cover)


In [ ]:
# ===== 3. 创建样本索引，不展开 [样本,180,特征] 巨型数组 =====
# target index = issue+lead; test targets require true valid day. input missing <=2 continuous days.
L=max(cfg.LEADS); H=cfg.HISTORY
missing=(~valid).astype(np.int32)
issue_indices=[];audit=[]
for i in range(H-1,len(cal)-L):
    if not valid[i]: continue
    win=missing[i-H+1:i+1]
    if win.mean()>cfg.MAX_MISSING_FRAC: continue
    if len(win) and np.max(np.diff(np.flatnonzero(np.r_[1,win==0,1]))-1)>cfg.MAX_INPUT_GAP: continue
    if any(not valid[i+d] for d in cfg.LEADS): continue
    if not (np.isfinite(F1[i-H+1:i+1]).all() and np.isfinite(F2[i-H+1:i+1]).all()): continue
    issue_indices.append(i)
ISS=np.asarray(issue_indices,np.int32)
if len(ISS)==0: raise RuntimeError('没有有效180→30/60/90 anchors，需检查缺日与冻结数据')
anchor=pd.DataFrame({'起报日期':cal[ISS]})
for l in cfg.LEADS: anchor[f'{l}天目标日期']=cal[ISS+l]
anchor.to_csv(DATA/'02_全部有效预测起报日期.csv',index=False,encoding='utf-8-sig')
print('anchor count=',len(ISS),' first=',cal[ISS[0]],'last=',cal[ISS[-1]])


In [ ]:
# ===== 4. 真正的 expanding-window 多年后报：test year = 2005/2009/2013/2017/2021/2024 =====
FOLDS=[];fr=[]; splitrows=[]
for fold_id,year in enumerate(cfg.TEST_YEARS,1):
    valstart=pd.Timestamp(year-1,1,1); teststart=pd.Timestamp(year,1,1)
    testend=pd.Timestamp(year,12,31)
    train_cut=valstart-pd.Timedelta(days=cfg.PURGE_DAYS)
    val_cut=teststart-pd.Timedelta(days=cfg.PURGE_DAYS)
    issues=cal[ISS]; latest=cal[ISS+L]
    tr=ISS[(latest<train_cut)]
    va=ISS[(issues>=valstart)&(issues<teststart)&(latest<val_cut)]
    te=ISS[(issues>=teststart)&(issues<=testend)]
    if len(tr)<cfg.MIN_TRAIN or len(va)<cfg.MIN_VAL or len(te)<cfg.MIN_TEST:
        raise RuntimeError(f'Fold {year} 样本不足: tr={len(tr)} val={len(va)} test={len(te)}；请查该年的日路径覆盖和连续缺日')
    if not(cal[tr+L].max()<cal[va].min()-pd.Timedelta(days=cfg.PURGE_DAYS) and cal[va+L].max()<cal[te].min()-pd.Timedelta(days=cfg.PURGE_DAYS)):
        raise RuntimeError(f'Fold {year} 发现严格时序泄漏')
    if (len(set(cal[tr+L])&set(cal[va+L])) or len(set(cal[tr+L])&set(cal[te+L])) or len(set(cal[va+L])&set(cal[te+L]))):
        raise RuntimeError(f'Fold {year} label date overlap')
    ff={'fold':fold_id,'year':year,'train':tr,'val':va,'test':te,'train_cut':train_cut,'val_cut':val_cut}
    FOLDS.append(ff)
    fr.append({'折次':fold_id,'测试年份':year,'训练样本':len(tr),'验证样本':len(va),'测试样本':len(te),
               '训练最晚标签日期':cal[tr+L].max(),'验证最早起报日期':cal[va].min(),
               '验证最晚标签日期':cal[va+L].max(),'测试最早起报日期':cal[te].min(),'测试最晚标签日期':cal[te+L].max(),'因果审计':'PASS'})
    for label,ix in [('训练',tr),('验证',va),('测试',te)]:
        splitrows.extend({'折次':fold_id,'测试年份':year,'集合':label,'起报日期':str(cal[i].date()),'最晚目标日期':str(cal[i+L].date())} for i in ix)
foldtab=pd.DataFrame(fr); foldtab.to_csv(DATA/'03_多年滚动后报折次及防泄漏审计.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(splitrows).to_csv(DATA/'03_逐样本时序划分.csv',index=False,encoding='utf-8-sig')
print('[STRICT MULTI-YEAR CAUSAL HINDCAST] ALL PASS'); display(foldtab)


In [ ]:
# ===== 6. Original-only：不生成DP，不修改冻结数据 =====
A2_ORIGINAL=A2.copy()
F2_ORIGINAL=F2.copy()
print('ORIGINAL FROZEN REPRESENTATION ONLY; dataset SHA256:',sign)


In [ ]:
# ===== 5. 每折 train-only 标准化 + 自适应 99% EOF =====
# 使用该fold训练期的独立日记录，而不是重叠180d窗口重复叠加做SVD；内存恒定在 O(feature²)
class DailyEOF:
    def fit(self,obs,target=0.99):
        obs=np.asarray(obs,np.float64)
        self.mu=np.mean(obs,axis=0); self.sd=np.std(obs,axis=0); self.sd[self.sd<1e-6]=1.
        z=(obs-self.mu)/self.sd
        cov=(z.T@z)/max(1,len(z)-1)
        eig,vec=np.linalg.eigh(cov)
        idx=np.argsort(eig)[::-1];eig=np.maximum(eig[idx],0);vec=vec[:,idx]
        rat=np.cumsum(eig)/(eig.sum()+1e-12); n=int(np.searchsorted(rat,target)+1)
        self.V=np.asarray(vec[:,:n],np.float32);self.explained=float(rat[n-1]);self.n=n
        return self
    def pc(self,x): return (((np.asarray(x)-self.mu)/self.sd)@self.V).astype(np.float32)
    def invert(self,pc): return ((np.asarray(pc)@self.V.T)*self.sd+self.mu).astype(np.float32)
    def z(self,x): return ((np.asarray(x)-self.mu)/self.sd).astype(np.float32)

def make_fold_data(f):
    train_days=(cal< f['train_cut']) & valid
    pp1=DailyEOF().fit(A1[train_days],cfg.EOF_VARIANCE_TARGET)
    pp2=DailyEOF().fit(A2[train_days],cfg.EOF_VARIANCE_TARGET)
    for pp in (pp1,pp2):
        if pp.explained<cfg.EOF_VARIANCE_TARGET-1e-6: raise RuntimeError('EOF 99%训练方差Gate未通过')
    # approx memory: ~17k days * 2*(31+277) floats; never expands 180 day windows
    z1=pp1.z(F1);z2=pp2.z(F2)
    pcs1=pp1.pc(F1);pcs2=pp2.pc(F2)
    ppath=CKPT/ACTIVE_REP/f'第{f["fold"]}折_{f["year"]}年'; ppath.mkdir(exist_ok=True,parents=True)
    np.savez_compressed(ppath/'训练期预处理参数.npz',
        b1_mu=pp1.mu,b1_sd=pp1.sd,b1_V=pp1.V,b2_mu=pp2.mu,b2_sd=pp2.sd,b2_V=pp2.V,
        branch1_lat=lat1,branch2_lon=lon2,train_label_cutoff=str(f['train_cut'].date()))
    return {'pp1':pp1,'pp2':pp2,'z1':z1,'z2':z2,'p1':pcs1,'p2':pcs2,'path':ppath}

# 只预审一折，防止全局重复保留大数组
_testFD=make_fold_data(FOLDS[0]);print('fold1 EOF modes:',_testFD['pp1'].n,_testFD['pp2'].n,'explained:',_testFD['pp1'].explained,_testFD['pp2'].explained)
del _testFD;gc.collect()
print('[TRAIN-ONLY EOF 99%] PASS')


In [ ]:
# ===== 6. 动态加载数据集，训练时每个batch只展开当前180天窗口 =====
class CausalWindowDataset(Dataset):
    def __init__(self,issues,scaled,pcs):
        self.issues=np.asarray(issues,np.int32);self.scaled=scaled;self.pcs=pcs
    def __len__(self): return len(self.issues)
    def __getitem__(self,j):
        i=int(self.issues[j]);z=self.scaled[i-H+1:i+1]
        dz=np.empty_like(z);dz[0]=0;dz[1:]=z[1:]-z[:-1]
        x=np.concatenate([z,dz],axis=-1)
        y=np.stack([self.pcs[i+d] for d in cfg.LEADS])
        return torch.from_numpy(x.copy()),torch.from_numpy(y.copy())

class StepEncoder(nn.Module):
    # 与旧原稿的per-step TCN/Conv编码保持一致：输入路径特征卷积，然后LSTM承担时间建模
    def __init__(self,feature,branch,kind='tcn'):
        super().__init__();self.kind=kind;self.drop=nn.Dropout(cfg.DROPOUT)
        if kind=='tcn':
            channels=16 if branch==1 else 32;kernel=3 if branch==1 else 5
            self.conv=nn.Conv1d(1,channels,kernel,dilation=2)
            self.bn=nn.BatchNorm1d(channels)
            conv_len=feature+2-2*(kernel-1)
            self.fc=nn.Linear(channels*conv_len,32 if branch==1 else 64)
            self.out_dim=self.fc.out_features
        elif kind=='conv':
            c1=16 if branch==1 else 32;c2=32 if branch==1 else 64
            self.c1=nn.Conv1d(1,c1,3,padding=1);self.c2=nn.Conv1d(c1,c2,3,padding=1)
            self.out_dim=c2*feature
        else: raise ValueError(kind)
    def forward(self,x):
        B,T,F=x.shape; z=x.reshape(B*T,1,F)
        if self.kind=='tcn':
            z=torch.nn.functional.pad(z,(2,0)) # 与R3原架构一致的左侧填充
            z=self.drop(torch.relu(self.bn(self.conv(z))))
            z=self.drop(torch.relu(self.fc(z.flatten(1))))
        else:
            z=self.drop(torch.relu(self.c1(z)));z=self.drop(torch.relu(self.c2(z)));z=z.flatten(1)
        return z.reshape(B,T,-1)

class Hybrid(nn.Module):
    def __init__(self,feature,pc_dim,branch,kind='tcn',recur='lstm'):
        super().__init__();self.enc=StepEncoder(feature,branch,kind)
        h=32 if branch==1 else 64
        R=nn.GRU if recur=='gru' else nn.LSTM
        self.rnn=R(self.enc.out_dim,h,batch_first=True)
        self.fc=nn.Linear(h,3*pc_dim);self.pc_dim=pc_dim
    def forward(self,x):
        h,_=self.rnn(self.enc(x));return self.fc(h[:,-1]).reshape(len(x),3,self.pc_dim)

class MatchedLSTM(nn.Module):
    def __init__(self,feature,pc_dim,hidden):
        super().__init__();self.rnn=nn.LSTM(feature,hidden,batch_first=True)
        self.fc=nn.Linear(hidden,3*pc_dim);self.pc_dim=pc_dim
    def forward(self,x):
        h,_=self.rnn(x);return self.fc(h[:,-1]).reshape(len(x),3,self.pc_dim)

def nparams(m): return sum(p.numel() for p in m.parameters())
def match_lstm(feature,pc_dim,branch):
    target=nparams(Hybrid(feature,pc_dim,branch))
    # Pytorch single-layer LSTM: 4h(feature+h+2) + 3pc*h + 3pc
    opts=range(8,1025)
    h=min(opts,key=lambda hh:abs(4*hh*(feature+hh+2)+(3*pc_dim)*(hh+1)-target))
    actual=4*h*(feature+h+2)+(3*pc_dim)*(h+1)
    if abs(actual-target)/target>0.08: raise RuntimeError('Matched-LSTM参数量相对差>8%，不满足公平基线')
    return h,target,actual

def make_net(name,branch,fd):
    feature=fd[f'z{branch}'].shape[1]*2;pdim=fd[f'pp{branch}'].n
    if name=='Matched-LSTM':
        h,_,_=match_lstm(feature,pdim,branch);return MatchedLSTM(feature,pdim,h)
    if name=='EOF-TCN-LSTM': return Hybrid(feature,pdim,branch,'tcn','lstm')
    if name=='EOF-Conv-LSTM': return Hybrid(feature,pdim,branch,'conv','lstm')
    if name=='EOF-Conv-GRU': return Hybrid(feature,pdim,branch,'conv','gru')
    if name=='EOF-TCN-GRU': return Hybrid(feature,pdim,branch,'tcn','gru')
    raise ValueError(name)




# ===== R5.3：路径坐标/一阶差分分为两个空间输入通道 =====
class ChannelSeparatedEncoder(nn.Module):
    """单时刻空间卷积；输入[B,T,2*M] => Conv1d [B*T,2,M]."""
    def __init__(self,feature,branch):
        super().__init__()
        assert feature%2==0, '需要[x, delta]等长拼接输入'
        self.raw_dim=feature//2
        base_c=16 if branch==1 else 32
        k=3 if branch==1 else 5
        channels=base_c*2
        # 正确的对称空间填充：dilation=2 -> padding=k-1，输出仍为M
        self.conv=nn.Conv1d(2,channels,kernel_size=k,dilation=2,padding=k-1)
        self.bn=nn.BatchNorm1d(channels)
        self.fc=nn.Linear(channels*self.raw_dim,32 if branch==1 else 64)
        self.out_dim=self.fc.out_features
        self.drop=nn.Dropout(cfg.DROPOUT)
    def channelize(self,x):
        B,T,F=x.shape
        if F!=self.raw_dim*2: raise RuntimeError('R53特征维不匹配')
        return torch.stack((x[:,:,:self.raw_dim],x[:,:,self.raw_dim:]),dim=2).reshape(B*T,2,self.raw_dim)
    def forward(self,x):
        B,T,F=x.shape
        z=self.channelize(x)
        z=self.drop(torch.relu(self.bn(self.conv(z))))
        if z.shape[2]!=self.raw_dim:raise RuntimeError('R53 spatial-conv输出宽度发生变化')
        z=self.drop(torch.relu(self.fc(z.flatten(1))))
        return z.reshape(B,T,-1)

class ChannelHybrid(nn.Module):
    def __init__(self,feature,pc_dim,branch,projection=None,use_pc_skip=False):
        super().__init__()
        self.enc=ChannelSeparatedEncoder(feature,branch)
        self.use_pc_skip=bool(use_pc_skip)
        self.raw_dim=feature//2
        self.pc_dim=pc_dim
        if self.use_pc_skip:
            self.register_buffer('V',torch.as_tensor(projection,dtype=torch.float32))
        hidden=32 if branch==1 else 64
        in_dim=self.enc.out_dim+(2*pc_dim if self.use_pc_skip else 0)
        self.rnn=nn.LSTM(in_dim,hidden,batch_first=True)
        self.fc=nn.Linear(hidden,3*pc_dim)
    def eof_skip(self,x):
        if not self.use_pc_skip: raise RuntimeError('no PC skip in this model')
        return torch.cat((x[:,:,:self.raw_dim]@self.V,x[:,:,self.raw_dim:]@self.V),dim=-1)
    def forward(self,x):
        z=self.enc(x)
        if self.use_pc_skip: z=torch.cat((z,self.eof_skip(x)),dim=-1)
        h,_=self.rnn(z)
        return self.fc(h[:,-1]).reshape(len(x),3,self.pc_dim)

R53_CANDIDATES=('R53-ChannelConv','R53-ChannelConv-PCConcat')
R53_BASELINES=('Matched-LSTM','EOF-TCN-LSTM','R53-CapacityMatchedLSTM')
R53_ALL=R53_BASELINES+R53_CANDIDATES

def make_r53_model(name,branch,fd):
    feature=fd[f'z{branch}'].shape[1]*2
    pdim=fd[f'pp{branch}'].n
    if name in ('Matched-LSTM','EOF-TCN-LSTM'):
        return make_net(name,branch,fd)
    if name=='R53-CapacityMatchedLSTM':
        ref=ChannelHybrid(feature,pdim,branch,fd[f'pp{branch}'].V,use_pc_skip=True)
        h,_,_=matched_hidden_for_target(feature,pdim,nparams(ref))
        return MatchedLSTM(feature,pdim,h)
    if name=='R53-ChannelConv':
        return ChannelHybrid(feature,pdim,branch)
    if name=='R53-ChannelConv-PCConcat':
        return ChannelHybrid(feature,pdim,branch,fd[f'pp{branch}'].V,use_pc_skip=True)
    raise ValueError(name)

def matched_hidden_for_target(feature,pdim,target):
    # 单层 PyTorch LSTM 参数解析式，替代遍历建立 1017 个网络实例
    def count(h):return 4*h*(feature+h+2)+3*pdim*(h+1)
    best=min(range(8,1025),key=lambda h:abs(count(h)-target))
    actual=count(best)
    ratio=abs(actual-target)/max(target,1)
    if ratio>0.08:raise RuntimeError(f'参数匹配误差过大：{ratio:.2%}')
    return best,actual,ratio

def model_loss(pred,target,fd,branch,name):
    # 与R5.2完全相同的训练目标：只用PC-MSE做结构归因。
    return torch.nn.functional.mse_loss(pred,target)


In [ ]:
# ===== 8. 物理公里误差：局地投影+最近点相邻线段的双向正交路径距离 =====
# 对于仅122-145E的分支2，重建得到共同定义域；不是对每一点简单经纬度平方误差。
# 下方是双向point-to-polyline近似cross-track，使用KDTree近邻三顶点及其邻接segments。
def merge_path(branch1,branch2):
    a=np.asarray(branch1,float);b=np.asarray(branch2,float)
    candidates=np.flatnonzero(lon2>a[-1]);
    k=int(candidates[np.argmin((lon2[candidates]-a[-1])**2+(b[candidates]-lat1[-1])**2)]) if len(candidates) else len(lon2)
    xs=np.r_[a,lon2[k:]];ys=np.r_[lat1,b[k:]]
    good=np.isfinite(xs)&np.isfinite(ys)
    return xs[good],ys[good]

def one_direction(A,B):
    # A,B: 2d coordinates (km); approximate nearest polyline segment for each A point
    if len(A)<2 or len(B)<2: return np.array([np.nan])
    tree=cKDTree(B);_,knn=tree.query(A,k=min(3,len(B)))
    knn=np.asarray(knn);knn=knn[:,None] if knn.ndim==1 else knn
    ix=np.unique(np.clip(np.concatenate([knn.ravel()-1,knn.ravel()]),0,len(B)-2))
    # segment shortlist globally may grow, but polynomial per path length (~300) and is vectorized
    v=B[ix+1]-B[ix];w=A[:,None,:]-B[ix][None,:,:]
    den=np.maximum(np.sum(v*v,axis=1),1e-12)
    t=np.clip(np.sum(w*v[None,:,:],axis=2)/den[None,:],0,1)
    q=B[ix][None,:,:]+t[:,:,None]*v[None,:,:]
    d=np.sqrt(np.sum((A[:,None,:]-q)**2,axis=2))
    return d.min(axis=1)

def errors_for_pair(pred1,pred2,truth1,truth2):
    x,y=merge_path(pred1,pred2);xt,yt=merge_path(truth1,truth2)
    lat0=np.mean(np.r_[y,yt]);lon0=np.mean(np.r_[x,xt]);c=np.cos(np.deg2rad(lat0))
    A=np.column_stack([(x-lon0)*111.32*c,(y-lat0)*111.32]);B=np.column_stack([(xt-lon0)*111.32*c,(yt-lat0)*111.32])
    d=np.r_[one_direction(A,B),one_direction(B,A)]
    return float(np.mean(d**2)),float(np.mean(d)),float(np.percentile(d,90))

def record_errors(fold,model,seed,issues,p1,p2):
    out=[]
    for n,i in enumerate(issues):
        for k,lead in enumerate(cfg.LEADS):
            y1=A1[i+lead];y2=A2[i+lead]
            mse,mean,p90=errors_for_pair(p1[n,k],p2[n,k],y1,y2)
            out.append({'折次':fold['fold'],'测试年份':fold['year'],'模型':model,'种子':seed,'起报日期':str(cal[i].date()),'目标日期':str(cal[i+lead].date()),'提前期_天':int(lead),
                        '横向路径误差MSE_km2':mse,'横向路径平均误差_km':mean,'横向路径P90_km':p90,
                        '分支1经度RMSE_度':float(np.sqrt(np.mean((p1[n,k]-y1)**2))),
                        '分支2纬度RMSE_度':float(np.sqrt(np.mean((p2[n,k]-y2)**2)))})
    return pd.DataFrame(out)


In [ ]:
# ===== 9. R5 训练封装、物理评价、可恢复 checkpoints =====
from contextlib import nullcontext

def r5_seed(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)

def run_predict(net, issues, fd, branch):
    ds=CausalWindowDataset(issues,fd[f'z{branch}'],fd[f'p{branch}'])
    ld=DataLoader(ds,batch_size=cfg.BATCH_SIZE,shuffle=False,num_workers=0)
    arr=[];net.eval()
    with torch.no_grad():
        for xx,_ in ld: arr.append(net(xx.to(DEV)).float().cpu().numpy())
    return np.concatenate(arr,axis=0)

def fit_r5_branch(split, fd, name, seed, branch, root_dir, max_epochs=None, patience=None):
    max_epochs=cfg.PILOT_EPOCHS if max_epochs is None else max_epochs
    patience=cfg.PILOT_PATIENCE if patience is None else patience
    folder=root_dir/ACTIVE_REP/f'QA_{split["year"]-1}'/name/f'seed_{seed}'
    folder.mkdir(parents=True,exist_ok=True)
    weight=folder/f'branch{branch}_best.pt'; historyfile=folder/f'branch{branch}_history.csv'
    latest=folder/f'branch{branch}_latest.pt'
    # best checkpoint 附带实验合同签名；仅同一fold/样本划分/网络版本可复用
    contract={'name':name,'seed':seed,'branch':branch,'tr_n':int(len(split['train'])),
              'val_n':int(len(split['val'])),'tr_last':int(split['train'][-1]),
              'val_first':int(split['val'][0]),'val_last':int(split['val'][-1]),'train_first':int(split['train'][0]),'max_epochs':max_epochs,
              'feature_dims':[fd['z1'].shape[1],fd['z2'].shape[1]],'version':'R5.7-original-stability-v1','rep':ACTIVE_REP,'dataset_signature':sign}
    contract_id=hashlib.sha256(json.dumps(contract,sort_keys=True).encode()).hexdigest()
    r5_seed(seed)
    net=make_r53_model(name,branch,fd).to(DEV)
    if weight.exists() and historyfile.exists() and not cfg.FORCE_TRAIN:
        ck=torch.load(weight,map_location='cpu',weights_only=True)
        if ck.get('contract_id')==contract_id:
            net.load_state_dict(ck['state']);return net,pd.read_csv(historyfile)
    ds_tr=CausalWindowDataset(split['train'],fd[f'z{branch}'],fd[f'p{branch}'])
    ds_va=CausalWindowDataset(split['val'],fd[f'z{branch}'],fd[f'p{branch}'])
    ld_tr=DataLoader(ds_tr,batch_size=cfg.BATCH_SIZE,shuffle=True,num_workers=0,pin_memory=True)
    ld_va=DataLoader(ds_va,batch_size=cfg.BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True)
    opt=torch.optim.AdamW(net.parameters(),lr=cfg.LR,weight_decay=cfg.WEIGHT_DECAY)
    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,factor=0.5,patience=4)
    amp=bool(cfg.USE_AMP)
    scaler=torch.amp.GradScaler('cuda',enabled=amp)
    start_epoch=1;best=float('inf');bestep=0;bad=0;hist=[]
    if latest.exists() and not cfg.FORCE_TRAIN:
        try:
            ck=torch.load(latest,map_location='cpu',weights_only=True)
            if ck.get('contract_id')==contract_id:
                net.load_state_dict(ck['state']);opt.load_state_dict(ck['opt']);scheduler.load_state_dict(ck['scheduler']);scaler.load_state_dict(ck['scaler'])
                start_epoch=int(ck['epoch'])+1;best=float(ck['best']);bestep=int(ck['bestep']);bad=int(ck['bad']);hist=ck['hist']
                print('[RESUME]',folder,'epoch',start_epoch)
        except Exception as e:print('[resume invalid]',e)
    for ep in range(start_epoch,max_epochs+1):
        net.train();ts=tn=0
        for x,y in ld_tr:
            x=x.to(DEV,non_blocking=True);y=y.to(DEV,non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda',enabled=amp):
                pred=net(x)
                loss=model_loss(pred.float(),y.float(),fd,branch,name)
            if not torch.isfinite(loss):raise RuntimeError(f'{name} B{branch}: non-finite loss')
            scaler.scale(loss).backward();scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(),1.0)
            scaler.step(opt);scaler.update();ts+=float(loss.detach())*len(x);tn+=len(x)
        net.eval();vs=vn=0
        with torch.no_grad():
            for x,y in ld_va:
                x=x.to(DEV);y=y.to(DEV)
                with torch.autocast('cuda',enabled=amp):
                    p=net(x);loss=model_loss(p.float(),y.float(),fd,branch,name)
                vs+=float(loss.detach())*len(x);vn+=len(x)
        val=vs/max(1,vn);scheduler.step(val)
        hist.append({'epoch':ep,'train_loss':ts/max(1,tn),'val_loss':val,'lr':opt.param_groups[0]['lr']})
        if val<best-1e-7:
            best=val;bestep=ep;bad=0
            torch.save({'state':{k:v.detach().cpu() for k,v in net.state_dict().items()},'contract_id':contract_id,'epoch':ep,'best_val':best},weight)
        else:bad+=1
        if ep%5==0 or bad>=patience or ep==max_epochs:
            torch.save({'state':{k:v.detach().cpu() for k,v in net.state_dict().items()},'opt':opt.state_dict(),
                        'scheduler':scheduler.state_dict(),'scaler':scaler.state_dict(),'epoch':ep,'best':best,
                        'bestep':bestep,'bad':bad,'hist':hist,'contract_id':contract_id},latest)
        if ep==1 or ep%10==0 or bad>=patience:print(f'{name} B{branch} seed{seed} ep{ep} train={hist[-1]["train_loss"]:.5f} val={val:.5f}')
        if bad>=patience:break
    pd.DataFrame(hist).to_csv(historyfile,index=False,encoding='utf-8-sig')
    if latest.exists():latest.unlink()
    ck=torch.load(weight,map_location='cpu',weights_only=True)
    net.load_state_dict(ck['state']);net.eval()
    return net,pd.DataFrame(hist)

def evaluate_issues(f,name,seed,fd,issues,bmodel1,bmodel2):
    return record_errors(f,name,seed,issues,bmodel1,bmodel2)

def get_predictions(name,split,fd,seed,folder,max_epochs=None,patience=None):
    branches=[]
    for branch in (1,2):
        net,hist=fit_r5_branch(split,fd,name,seed,branch,folder,max_epochs,patience)
        pr=run_predict(net,split.get('qa',split.get('test')),fd,branch)
        branches.append(fd[f'pp{branch}'].invert(pr))
        del net,pr;torch.cuda.empty_cache()
    return branches


In [ ]:
# ===== 7. 预先固定 QA 拆分：标签与早停严格先于 QA 日期 =====
PILOT_QA_YEARS = (2008,2016)
CONFIRM_QA_YEARS = (2010,2018)
R57_MODELS = ('Matched-LSTM','R53-CapacityMatchedLSTM','EOF-TCN-LSTM','R53-ChannelConv-PCConcat')
R57_SEEDS=tuple(cfg.PILOT_SEEDS)

# QA 年份不再依赖 R56 TEST_YEARS 白名单；其余合同完全一致。
def r57_pilot_split(qa_year):
    qa_start=pd.Timestamp(qa_year,1,1)
    next_start=pd.Timestamp(qa_year+1,1,1)
    inner_start=pd.Timestamp(qa_year-1,1,1)
    purge=pd.Timedelta(days=cfg.PURGE_DAYS)
    train=ISS[cal[ISS+L] < inner_start-purge]
    val=ISS[(cal[ISS]>=inner_start) & (cal[ISS+L]<qa_start-purge)]
    qa=ISS[(cal[ISS]>=qa_start) & (cal[ISS+L]<next_start-purge)]
    if min(len(train),len(val),len(qa))<min(cfg.MIN_TRAIN,cfg.MIN_VAL):
        raise RuntimeError(f'QA {qa_year} 样本量不足 train={len(train)} val={len(val)} qa={len(qa)}')
    if not (len(train)>=cfg.MIN_TRAIN and len(val)>=cfg.MIN_VAL and len(qa)>=cfg.MIN_VAL):
        raise RuntimeError(f'QA {qa_year} 数据资格未满足')
    assert cal[train+L].max()<cal[val].min()-purge
    assert cal[val+L].max()<cal[qa].min()-purge
    return {'fold':qa_year,'year':qa_year+1,'train':train,'val':val,'qa':qa,'train_cut':inner_start-purge}

PILOTS=[r57_pilot_split(y) for y in PILOT_QA_YEARS]
CONFIRMS=[r57_pilot_split(y) for y in CONFIRM_QA_YEARS]
SPLIT_ROWS=[]
for phase,fs in [('资格探索',PILOTS),('后续年份确认',CONFIRMS)]:
    for f in fs:
        SPLIT_ROWS.append({'阶段':phase,'QA年份':f['year']-1,'训练数':len(f['train']),'早停数':len(f['val']),'QA数':len(f['qa']),
          '训练最晚标签日期':str(cal[f['train']+L].max().date()),'早停最晚标签日期':str(cal[f['val']+L].max().date()),
          'QA最早起报日期':str(cal[f['qa']].min().date())})
split_df=pd.DataFrame(SPLIT_ROWS)
split_df.to_csv(DATA/'R57_01_严格因果时序拆分.csv',index=False,encoding='utf-8-sig')
display(split_df)
print('[R57 STRICT CAUSAL SPLIT] PASS')


In [ ]:
# ===== R59. 自动归档：失败也可独立执行 =====
from IPython.display import FileLink, display, Markdown
import zipfile, traceback, hashlib, json
SHARE_EXT={'.csv','.json','.txt','.md','.png','.pdf','.log'}

def archive_r59(stage, problem=None):
    (REP/'R59_运行状态.json').write_text(json.dumps({
        'stage':stage,'utc':pd.Timestamp.now(tz='UTC').isoformat(),
        'dataset_sha256':sign,'pilot_years':[2008,2016],
        'confirmation_years':[2010,2018],'seeds':list(cfg.PILOT_SEEDS),
        'teacher_parent':'R57 Matched-LSTM','exception':problem,
        'not_blind_test':True},indent=2,ensure_ascii=False),encoding='utf-8')
    files=[f for d in [DATA,REP,FZH,FEN] for f in d.rglob('*')
           if f.is_file() and f!=REP/'R59_文件与SHA256清单.csv'
           and f.suffix.lower() in SHARE_EXT and f.stat().st_size<=100*1024*1024]
    rows=[{'file':str(f.relative_to(OUT)),'bytes':f.stat().st_size,
           'sha256':hashlib.sha256(f.read_bytes()).hexdigest()} for f in sorted(set(files))]
    manifest=REP/'R59_文件与SHA256清单.csv'
    pd.DataFrame(rows).to_csv(manifest,index=False,encoding='utf-8-sig')
    files.append(manifest)
    temp=ARCHIVE_PATH.with_suffix('.zip.tmp')
    try:
        with zipfile.ZipFile(temp,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
            for f in sorted(set(files)):
                z.write(f,arcname='R59_结果/'+str(f.relative_to(OUT)))
            z.writestr('R59_结果/README.txt','R59 研发资格数据，不代表盲测。权重及大规模父模型预测缓存未包含。')
        with zipfile.ZipFile(temp) as z:
            if z.testzip() is not None:raise IOError('zip test failed')
        temp.replace(ARCHIVE_PATH)
        digest=hashlib.sha256(ARCHIVE_PATH.read_bytes()).hexdigest()
        ARCHIVE_SHA.write_text(f'{digest}  {ARCHIVE_PATH.name}\n',encoding='utf-8')
        print('[R59 ARCHIVE]',stage,ARCHIVE_PATH,'size MB',round(ARCHIVE_PATH.stat().st_size/1048576,2))
        display(FileLink(str(ARCHIVE_PATH)))
        display(Markdown('WSL 本地文件：`'+str(ARCHIVE_PATH)+'`；Windows 的 D 盘根目录下可找到同名 ZIP。'))
        return ARCHIVE_PATH
    finally:
        if temp.exists():temp.unlink()

print('[R59 ARCHIVER READY]')


In [ ]:
# ===== R59. 分支耦合低秩校正器：不改父模型 LSTM 权重或预测端点 =====
import torch.nn.functional as F
from torch.utils.data import TensorDataset

class CrossBranchResidual(nn.Module):
    def __init__(self,pdim1,pdim2,hidden=24,mode='cross'):
        super().__init__()
        if mode not in ('cross','own'):raise ValueError(mode)
        self.mode=mode;self.p1=pdim1;self.p2=pdim2
        if mode=='cross':
            self.shared=nn.Sequential(nn.Linear(4*(pdim1+pdim2)+3,hidden),nn.SiLU(),
                                      nn.Linear(hidden,hidden),nn.SiLU(),
                                      nn.Linear(hidden,pdim1+pdim2))
            nn.init.zeros_(self.shared[-1].weight);nn.init.zeros_(self.shared[-1].bias)
        else:
            self.branch_heads=nn.ModuleList()
            for pdim in (pdim1,pdim2):
                head=nn.Sequential(nn.Linear(4*pdim+3,hidden),nn.SiLU(),
                                   nn.Linear(hidden,pdim))
                nn.init.zeros_(head[-1].weight);nn.init.zeros_(head[-1].bias)
                self.branch_heads.append(head)
    def forward(self,t1,t2,s1,s2,v7_1,v7_2,v30_1,v30_2,calendar):
        # [batch,3,pdim] future teacher PCs; [batch,pdim] current + 7/30day historical tendency
        B,K,_=t1.shape
        b1=s1[:,None,:].expand(-1,K,-1);b2=s2[:,None,:].expand(-1,K,-1)
        w1=v7_1[:,None,:].expand(-1,K,-1);w2=v7_2[:,None,:].expand(-1,K,-1)
        m1=v30_1[:,None,:].expand(-1,K,-1);m2=v30_2[:,None,:].expand(-1,K,-1)
        if self.mode=='cross':
            x=torch.cat([t1,t2,b1,b2,w1,w2,m1,m2,calendar],dim=-1)
            corr=self.shared(x)
            c1,c2=torch.split(corr,[self.p1,self.p2],dim=-1)
        else:
            a=torch.cat([t1,b1,w1,m1,calendar],dim=-1)
            b=torch.cat([t2,b2,w2,m2,calendar],dim=-1)
            c1=self.branch_heads[0](a);c2=self.branch_heads[1](b)
        return t1+c1,t2+c2,c1,c2

def make_features(fd,issues,t1,t2):
    issues=np.asarray(issues,dtype=int)
    a=fd['p1'];b=fd['p2']
    if issues.min()<30: raise RuntimeError('历史差分窗口不足30天')
    day=np.array([cal[i].dayofyear for i in issues],np.float32)
    doy=np.stack([
      np.stack([np.sin(2*np.pi*(day+lead)/365.2425),np.cos(2*np.pi*(day+lead)/365.2425),
                np.full_like(day,lead/90.0)],axis=1) for lead in cfg.LEADS
    ],axis=1).astype('float32')
    y1=np.stack([a[issues+lead] for lead in cfg.LEADS],axis=1).astype('float32')
    y2=np.stack([b[issues+lead] for lead in cfg.LEADS],axis=1).astype('float32')
    arrays=[t1,t2,a[issues],b[issues],a[issues]-a[issues-7],b[issues]-b[issues-7],
            a[issues]-a[issues-30],b[issues]-b[issues-30],doy,y1,y2]
    if not all(np.isfinite(x).all() for x in arrays):raise RuntimeError('校正器输入/标签包含NaN')
    return tuple(torch.from_numpy(np.asarray(x,np.float32)) for x in arrays)

def loss_phys_pc(c1,c2,y1,y2,fd,reg1=None,reg2=None):
    # 训练期EOF坐标反投影，局地经度换算 + 纬度换算；共享一套km物理尺度
    phys=[]
    for cp,yp,br in ((c1,y1,1),(c2,y2,2)):
        pp=fd[f'pp{br}']
        V=torch.as_tensor(pp.V,device=cp.device,dtype=torch.float32)
        sd=torch.as_tensor(pp.sd,device=cp.device,dtype=torch.float32)
        km=111.32*np.cos(np.deg2rad(float(np.mean(lat1)))) if br==1 else 111.32
        phys.append((((cp-yp)@V.T)*sd*km).square().mean())
    pc=.5*(F.mse_loss(c1,y1)+F.mse_loss(c2,y2))
    phy=.5*(phys[0]+phys[1]) / (50.0**2)
    reg=(reg1.square().mean()+reg2.square().mean())*.005 if reg1 is not None else 0.
    return pc*.65+phy*.35+reg

# 额外参数量不能依赖测试表现；计算 LSTM 实际参数量，并与更大容量 LSTM 对照
FD_TEST=make_fold_data(PILOTS[0])
n1=make_r53_model('Matched-LSTM',1,FD_TEST);n2=make_r53_model('Matched-LSTM',2,FD_TEST)
ref1=make_r53_model('R53-CapacityMatchedLSTM',1,FD_TEST);ref2=make_r53_model('R53-CapacityMatchedLSTM',2,FD_TEST)
hy1=make_r53_model('R53-ChannelConv-PCConcat',1,FD_TEST);hy2=make_r53_model('R53-ChannelConv-PCConcat',2,FD_TEST)
pt1=FD_TEST['pp1'].n;pt2=FD_TEST['pp2'].n
rows=[]
for mode in ('own','cross'):
    m=CrossBranchResidual(pt1,pt2,hidden=24,mode=mode)
    ct=nparams(n1)+nparams(n2)+nparams(m);baseline=nparams(n1)+nparams(n2)
    rows.append({'mode':mode,'LSTM_parent_params':baseline,'residual_params':nparams(m),
        'total_trainable_plus_parent':ct,'overhead_percent':100*(ct/baseline-1),
        'larger_LSTM_params':nparams(ref1)+nparams(ref2)})
    if ct>=nparams(ref1)+nparams(ref2):raise RuntimeError('新模型总参数超过更大的LSTM基线，容量审计不公平')
    if (ct/baseline-1)>0.06:raise RuntimeError('新增参数超过6%，不属于轻量改进')
hybrid_total=nparams(hy1)+nparams(hy2)+nparams(CrossBranchResidual(pt1,pt2,mode='cross'))
large_lstm_params=nparams(ref1)+nparams(ref2)
rows.append({'mode':'hybrid_cross','LSTM_parent_params':nparams(hy1)+nparams(hy2),
   'residual_params':nparams(CrossBranchResidual(pt1,pt2,mode='cross')),
   'total_trainable_plus_parent':hybrid_total,
   'overhead_percent':100*(hybrid_total/(nparams(hy1)+nparams(hy2))-1),
   'larger_LSTM_params':large_lstm_params})
if abs(hybrid_total-large_lstm_params)/large_lstm_params>0.05:
   raise RuntimeError('R53混合候选与较大LSTM容量差异超过5%；需重新做容量对照')
pd.DataFrame(rows).to_csv(DATA/'R59_02_双分支参数量审计.csv',index=False,encoding='utf-8-sig')
display(pd.DataFrame(rows))
# 合成检查：初始化时严格等于父LSTM输出；改未来日历不能影响更早预测输入
for mode in ('own','cross'):
    mdl=CrossBranchResidual(pt1,pt2,mode=mode).to(DEV)
    data=[torch.randn(3,3,pt1,device=DEV),torch.randn(3,3,pt2,device=DEV)]
    st=[torch.randn(3,pt1,device=DEV),torch.randn(3,pt2,device=DEV)]
    args=data+st+[torch.zeros_like(st[0]),torch.zeros_like(st[1]),torch.zeros_like(st[0]),torch.zeros_like(st[1]),torch.randn(3,3,3,device=DEV)]
    z1,z2,dc1,dc2=mdl(*args)
    assert torch.allclose(z1,data[0],atol=0,rtol=0) and torch.allclose(z2,data[1],atol=0,rtol=0)
    (z1.sum()+z2.sum()).backward()
    assert all(p.grad is not None for p in mdl.parameters())
    del mdl
print('[R59 MODEL SHAPE/IDENTITY/GRAD/CAPACITY] PASS')
del FD_TEST,n1,n2,ref1,ref2,hy1,hy2;gc.collect();torch.cuda.empty_cache()


In [ ]:
# ===== R59. 冻结父模型：优先只读复用R57中已验证权重，缺失时本地同合同补训 =====
PILOT_QA_YEARS=(2008,2016)
CONFIRM_QA_YEARS=(2010,2018)
R59_SEEDS=tuple(cfg.PILOT_SEEDS)
R59_MODES=('own','cross')
TEACHER_NAMES=('Matched-LSTM','R53-CapacityMatchedLSTM','R53-ChannelConv-PCConcat')
CACHED=OUT/'教师预测缓存';CACHED.mkdir(parents=True,exist_ok=True)
CORR_CKPT=CKPT/'轻量校正器';CORR_CKPT.mkdir(parents=True,exist_ok=True)

def parent_contract(f,fd,name,seed,branch):
    c={'name':name,'seed':seed,'branch':branch,'tr_n':int(len(f['train'])),
        'val_n':int(len(f['val'])),'tr_last':int(f['train'][-1]),
        'val_first':int(f['val'][0]),'val_last':int(f['val'][-1]),
        'train_first':int(f['train'][0]),'max_epochs':cfg.PILOT_EPOCHS,
        'feature_dims':[fd['z1'].shape[1],fd['z2'].shape[1]],
        'version':'R5.7-original-stability-v1','rep':ACTIVE_REP,'dataset_signature':sign}
    return hashlib.sha256(json.dumps(c,sort_keys=True).encode()).hexdigest()

def get_parent(f,fd,name,seed,branch):
    pt=(PARENT_R57/ACTIVE_REP/f'QA_{f["year"]-1}'/name/f'seed_{seed}'/
         f'branch{branch}_best.pt')
    net=make_r53_model(name,branch,fd)
    if pt.exists() and not cfg.FORCE_TRAIN:
        ck=torch.load(pt,map_location='cpu',weights_only=True)
        if ck.get('contract_id')==parent_contract(f,fd,name,seed,branch):
            net.load_state_dict(ck['state']);net=net.to(DEV);net.eval()
            print('[R59 READONLY PARENT]',name,'QA',f['year']-1,'seed',seed,'B',branch)
            return net
        else:
            print('[R59] 旧权重合同签名不匹配，重新训练父模型，不读取旧参数',pt)
    net,_=fit_r5_branch(f,fd,name,seed,branch,CKPT/'父模型_缺失时补训',cfg.PILOT_EPOCHS,cfg.PILOT_PATIENCE)
    return net

def prepare_teacher(f,fd,seed,name):
    suffix=hashlib.sha256(json.dumps({'sig':sign,'qa':f['year']-1,'seed':seed,'name':name,
      'ntrain':len(f['train']),'nval':len(f['val']),'nqa':len(f['qa']),
      'epoch':cfg.PILOT_EPOCHS,'version':'R59-001'},sort_keys=True).encode()).hexdigest()[:16]
    file=CACHED/f'qa{f["year"]-1}_{name}_seed{seed}_{suffix}.npz'
    if file.exists() and not cfg.FORCE_TRAIN:
        with np.load(file,allow_pickle=False) as z:
            output={k:np.asarray(z[k],np.float32) for k in ('tr_b1','tr_b2','val_b1','val_b2','qa_b1','qa_b2')}
            for lab,split_name in (('tr','train'),('val','val'),('qa','qa')):
                if len(output[f'{lab}_b1'])!=len(f[split_name]):raise RuntimeError('教师预测缓存数量与拆分不匹配')
        print('[R59 RESUME TEACHER]',file.name)
        return output
    output={}
    for branch in (1,2):
        net=get_parent(f,fd,name,seed,branch)
        for lab,spl in [('tr','train'),('val','val'),('qa','qa')]:
            output[f'{lab}_b{branch}']=run_predict(net,f[spl],fd,branch).astype('float32')
        del net;torch.cuda.empty_cache()
    np.savez_compressed(file,**output)
    return output

# 仅训练强LSTM教师；较大容量LSTM保持R57独立对照，不给校正器使用其未来QA信息
print('[R59 PARENT LOADER READY]')


In [ ]:
# ===== R59. 可追溯的第二阶段残差校正训练（严格仅用训练、早停期） =====
R59_EPOCHS=22
R59_PATIENCE=5
R59_LR=0.0015
R59_MIN_DEV_GAIN=0.001  # 内层验证综合loss至少改善0.1%；允许原样回退
R59_MAX_TRAIN_ISSUES=1800 # 优先利用最近多个训练年份，避免全部旧年代拟合纠偏器

def tensors_for(fd,issues,pr):
    return make_features(fd,issues,pr['b1'],pr['b2'])

def _metric_on_tensor(model,tensors,fd,device=DEV):
    model.eval();tn=ss=0.;
    with torch.no_grad():
        for start in range(0,len(tensors[0]),128):
            b=[t[start:start+128].to(device) for t in tensors]
            c1,c2,dc1,dc2=model(*b[:9])
            ss+=float(loss_phys_pc(c1,c2,b[9],b[10],fd,dc1,dc2))*len(b[0]);tn+=len(b[0])
    return ss/max(1,tn)

def _teacher_loss(tensors,fd):
    total=0.;n=0
    for i in range(0,len(tensors[0]),128):
        b=[t[i:i+128].to(DEV) for t in tensors]
        with torch.no_grad():val=loss_phys_pc(b[0],b[1],b[9],b[10],fd)
        total+=float(val)*len(b[0]);n+=len(b[0])
    return total/max(1,n)

def fit_correction(f,fd,seed,mode,teacher,teacher_name):
    train_idx=f['train'][-R59_MAX_TRAIN_ISSUES:]
    trp={'b1':teacher['tr_b1'][-len(train_idx):], 'b2':teacher['tr_b2'][-len(train_idx):]}
    vap={'b1':teacher['val_b1'],'b2':teacher['val_b2']}
    tr=tensors_for(fd,train_idx,trp)
    va=tensors_for(fd,f['val'],vap)
    baseline_val=_teacher_loss(va,fd)
    folder=CORR_CKPT/f'QA_{f["year"]-1}'/teacher_name/mode/f'seed_{seed}'
    folder.mkdir(parents=True,exist_ok=True)
    ckpath=folder/'best.pt'
    contract=hashlib.sha256(json.dumps({
        'ver':'R59-lowrank-v2','dataset_sha256':sign,'qa':f['year']-1,'teacher':teacher_name,
        'mode':mode,'seed':seed,'train_last':int(f['train'][-1]),'val_last':int(f['val'][-1]),
        'epochs':R59_EPOCHS,'lr':R59_LR,'p1':fd['pp1'].n,'p2':fd['pp2'].n,
        'val_loss':round(baseline_val,10)},sort_keys=True).encode()).hexdigest()
    r5_seed(seed+999) # 固定随机初始化必须在建模之前
    model=CrossBranchResidual(fd['pp1'].n,fd['pp2'].n,mode=mode).to(DEV)
    candidate_params=nparams(model)
    if ckpath.exists() and not cfg.FORCE_TRAIN:
        ck=torch.load(ckpath,map_location='cpu',weights_only=True)
        if ck.get('contract_id')==contract:
            model.load_state_dict(ck['state']);model.eval()
            return model,{'year':f['year']-1,'seed':seed,'mode':mode,'parent_model':teacher_name,
               'selected_using_inner_val_only':bool(ck['use']),
               'best_epoch':int(ck['best_epoch']),
               'val_improvement_fraction':float((ck['parent_val']-ck['best_val'])/max(1e-9,ck['parent_val'])),
               'val_teacher_loss':float(ck['parent_val']),'val_best_loss':float(ck['best_val']),
               'residual_params':candidate_params,'checkpoint_reused':True}
    # 新模块初始完全继承父模型输出；即使学习失败也绝不从QA期选择alpha
    loader=DataLoader(TensorDataset(*tr),batch_size=128,shuffle=True,num_workers=0,pin_memory=True)
    opt=torch.optim.AdamW(model.parameters(),lr=R59_LR,weight_decay=0.003)
    patience=0;best=baseline_val;best_ep=0;hist=[];best_state=None
    for ep in range(1,R59_EPOCHS+1):
        model.train();ls=nn_=0.
        for batch in loader:
            b=[x.to(DEV) for x in batch]; opt.zero_grad(set_to_none=True)
            p1,p2,dc1,dc2=model(*b[:9]);loss=loss_phys_pc(p1,p2,b[9],b[10],fd,dc1,dc2)
            if not torch.isfinite(loss):raise RuntimeError('残差校正loss非有限值')
            loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            opt.step();ls+=float(loss.detach())*len(b[0]);nn_+=len(b[0])
        val=_metric_on_tensor(model,va,fd)
        hist.append({'epoch':ep,'train_loss':ls/max(1,nn_),'val_loss':val,'baseline_val_loss':baseline_val})
        if val<best-1e-6:
            best=val;best_ep=ep;patience=0
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        else:patience+=1
        if ep==1 or ep%5==0 or patience>=R59_PATIENCE:
            print('[R59 corrector]',f['year']-1,seed,mode,ep,'val',round(val,5),'parent',round(baseline_val,5))
        if patience>=R59_PATIENCE:break
    gain=(baseline_val-best)/max(1e-9,baseline_val)
    use=bool(best_state is not None and gain>=R59_MIN_DEV_GAIN)
    if use:model.load_state_dict(best_state)
    else:
        model=CrossBranchResidual(fd['pp1'].n,fd['pp2'].n,mode=mode).to(DEV) # zero residual => parent
    model.eval()
    pd.DataFrame(hist).to_csv(folder/'training.csv',index=False,encoding='utf-8-sig')
    torch.save({'state':{k:v.detach().cpu() for k,v in model.state_dict().items()},
                'qa_year':f['year']-1,'mode':mode,'seed':seed,'use':use,
                'parent_val':baseline_val,'best_val':best,'best_epoch':best_ep,
                'dataset_sha256':sign,'teacher_name':teacher_name,'contract_id':contract},ckpath)
    status={'year':f['year']-1,'seed':seed,'mode':mode,'parent_model':teacher_name,'selected_using_inner_val_only':use,
            'best_epoch':best_ep,'val_improvement_fraction':float(gain),
            'val_teacher_loss':float(baseline_val),'val_best_loss':float(best),'residual_params':candidate_params}
    return model,status

def predict_correction(fd,f,teacher,model):
    arrays=tensors_for(fd,f['qa'],{'b1':teacher['qa_b1'],'b2':teacher['qa_b2']})
    pp1=[];pp2=[]
    model.eval()
    with torch.no_grad():
        for ix in range(0,len(arrays[0]),128):
            b=[t[ix:ix+128].to(DEV) for t in arrays]
            t1,t2,_,_=model(*b[:9]);pp1.append(t1.cpu().numpy());pp2.append(t2.cpu().numpy())
    return np.concatenate(pp1,axis=0),np.concatenate(pp2,axis=0)

print('[R59 SECOND STAGE TRAINING READY]')


In [ ]:
# ===== R59. 第一阶段 QA：公平比较父LSTM、容量更大LSTM、两种校正 =====
QA_FILE=DATA/'R59_03_共同冻结流轴逐样本误差.csv'
SELECT_FILE=DATA/'R59_03_内层验证选择记录.csv'
MODELS=('Matched-LSTM','R53-CapacityMatchedLSTM','R53-ChannelConv-PCConcat','R59-OwnBranch','R59-CrossBranch','R59-HybridCoupled')

def report_predictions(f,name,seed,p1,p2,phase):
    result=[]
    for n,i in enumerate(f['qa']):
        i=int(i)
        for k,lead in enumerate(cfg.LEADS):
            y1=A1[i+lead];y2=A2[i+lead]
            e,ma,p90=errors_for_pair(p1[n,k],p2[n,k],y1,y2)
            result.append({'阶段':phase,'QA年份':int(f['year']-1),'模型':name,'种子':int(seed),
                '起报日期':str(cal[i].date()),'目标日期':str(cal[i+lead].date()),
                '提前期_天':int(lead),'冻结目标MSE_km2':e,'冻结目标平均距离_km':ma,'冻结目标P90_km':p90})
    return pd.DataFrame(result)

def get_existing_predictions():
    if QA_FILE.exists():return pd.read_csv(QA_FILE)
    return pd.DataFrame()

def run_r59_phase(folds,phase):
    old=get_existing_predictions()
    old=old[old['阶段']!=phase] if len(old) else pd.DataFrame()
    rows=[];selector=[]
    for f in folds:
        fd=make_fold_data(f)
        try:
            for seed in R59_SEEDS:
                for parentname in TEACHER_NAMES:
                    teacher=prepare_teacher(f,fd,seed,parentname)
                    p1=fd['pp1'].invert(teacher['qa_b1']);p2=fd['pp2'].invert(teacher['qa_b2'])
                    rows.append(report_predictions(f,parentname,seed,p1,p2,phase))
                    # 先落盘，长实验即使中断也可恢复父模型的有效结果
                    pd.concat(([old] if len(old) else [])+rows,ignore_index=True).to_csv(QA_FILE,index=False,encoding='utf-8-sig')
                    if parentname in ('Matched-LSTM','R53-ChannelConv-PCConcat'):
                        modes=R59_MODES if parentname=='Matched-LSTM' else ('cross',)
                        for mode in modes:
                            model,status=fit_correction(f,fd,seed,mode,teacher,parentname)
                            c1,c2=predict_correction(fd,f,teacher,model)
                            name=({'own':'R59-OwnBranch','cross':'R59-CrossBranch'}[mode]
                                  if parentname=='Matched-LSTM' else 'R59-HybridCoupled')
                            rows.append(report_predictions(f,name,seed,fd['pp1'].invert(c1),fd['pp2'].invert(c2),phase))
                            selector.append({'阶段':phase,**status})
                            pd.concat(([old] if len(old) else [])+rows,ignore_index=True).to_csv(QA_FILE,index=False,encoding='utf-8-sig')
                            pd.DataFrame(selector).to_csv(SELECT_FILE,index=False,encoding='utf-8-sig')
                            print('[R59 QA]',phase,'year',f['year']-1,'seed',seed,'candidate',name,'len',len(f['qa']))
                            del model
                    del teacher;gc.collect();torch.cuda.empty_cache()
        finally:
            del fd;gc.collect();torch.cuda.empty_cache()
        archive_r59('after_QA_'+str(f['year']-1))
    return pd.concat(([old] if len(old) else [])+rows,ignore_index=True)

def r59_gate(phase,expected_years):
    raw=pd.read_csv(QA_FILE)
    d=raw[raw['阶段']==phase].copy()
    keys={(year,model,seed,lead) for year in expected_years for model in MODELS
          for seed in R59_SEEDS for lead in cfg.LEADS}
    actual=set(zip(d.QA年份.astype(int),d['模型'],d['种子'].astype(int),d['提前期_天'].astype(int)))
    if keys!=actual:raise RuntimeError('资格结果不完整：缺少或多出组 '+repr(list(keys-actual)[:8]))
    fold_map={f['year']-1:len(f['qa']) for f in PILOTS+CONFIRMS}
    group_n=d.groupby(['QA年份','模型','种子','提前期_天']).size()
    for (y,m,s,l),n in group_n.items():
        if n!=fold_map[y]:raise RuntimeError(f'QA({y},{m},{s},{l})记录数{n}不等于{fold_map[y]}')
    sums=(d.groupby(['QA年份','模型','种子','提前期_天'],as_index=False)
            .agg(RMSE_km=('冻结目标MSE_km2',lambda x:float(np.sqrt(np.mean(x))))))
    sums.to_csv(DATA/f'R59_04_{phase}_组别RMSE.csv',index=False,encoding='utf-8-sig')
    agg=sums.groupby(['模型','提前期_天'],as_index=False).RMSE_km.mean()
    agg.to_csv(DATA/f'R59_04_{phase}_模型汇总.csv',index=False,encoding='utf-8-sig')
    w=sums.pivot(index=['QA年份','种子','提前期_天'],columns='模型',values='RMSE_km')
    # 与固定的两个实际LSTM网络分别比较，不按每个QA单元挑选更小误差作为独立模型
    for candidate in ('R59-OwnBranch','R59-CrossBranch','R59-HybridCoupled'):
        w[candidate+'相对父LSTM改善_km']=w['Matched-LSTM']-w[candidate]
        w[candidate+'相对大容量LSTM改善_km']=w['R53-CapacityMatchedLSTM']-w[candidate]
    w.reset_index().to_csv(DATA/f'R59_05_{phase}_严格成对比较.csv',index=False,encoding='utf-8-sig')
    decision=[]
    for name in ('R59-OwnBranch','R59-CrossBranch','R59-HybridCoupled'):
        col=name+'相对大容量LSTM改善_km';gain=w[col]
        byyear=gain.groupby(level='QA年份').mean();bylead=gain.groupby(level='提前期_天').mean()
        byseedyear=gain.groupby(level=['QA年份','种子']).mean()
        mean=float(gain.mean())
        ok=(mean>=.5 and bool((byyear>0).all()) and bool((bylead>=0).all())
            and float((byseedyear>0).mean())>=2/3
            and (name!='R59-HybridCoupled' or float((w['R53-ChannelConv-PCConcat']-w[name]).mean())>0))
        decision.append({'阶段':phase,'模型':name,'大容量LSTM平均改善_km':mean,
            '父LSTM平均改善_km':float(w[name+'相对父LSTM改善_km'].mean()),
            '原R53混合模型改善_km':float((w['R53-ChannelConv-PCConcat']-w[name]).mean()),
            '各年份改善':{str(k):float(v) for k,v in byyear.items()},
            '各提前期改善':{str(k):float(v) for k,v in bylead.items()},
            '年份种子胜率':float((byseedyear>0).mean()),'Gate通过':bool(ok)})
    table=pd.DataFrame([{'阶段':v['阶段'],'模型':v['模型'],
             '大容量LSTM平均改善_km':v['大容量LSTM平均改善_km'],
             '父LSTM平均改善_km':v['父LSTM平均改善_km'],
             '年份种子胜率':v['年份种子胜率'],'Gate通过':v['Gate通过']} for v in decision])
    table.to_csv(DATA/f'R59_05_{phase}_资格判定.csv',index=False,encoding='utf-8-sig')
    passed=[v for v in decision if v['Gate通过']]
    selected=max(passed,key=lambda x:x['大容量LSTM平均改善_km'])['模型'] if passed else None
    result={'phase':phase,'selected':selected,'passed':bool(selected),'all_candidates':decision,
            'predeclared_gate':'avg_gain>=0.5 km vs fixed larger LSTM; all years>0; all leads>=0; year_seed_win>=2/3',
            'prior_QA_exposed':True}
    (REP/f'R59_05_{phase}_资格判定.json').write_text(json.dumps(result,indent=2,ensure_ascii=False),encoding='utf-8')
    display(agg.pivot(index='模型',columns='提前期_天',values='RMSE_km'))
    display(table)
    return result

print('[R59 EVAL ENGINE READY]')


In [ ]:
# ===== R59. 探索 QA：仍使用研发用年份，不将探索称为独立测试 =====
try:
    qa=run_r59_phase(PILOTS,'探索QA')
    R59_EXPLORATORY=r59_gate('探索QA',PILOT_QA_YEARS)
    archive_r59('exploratory_finished')
except Exception:
    msg=traceback.format_exc()
    (REP/'R59_异常回溯.txt').write_text(msg,encoding='utf-8')
    archive_r59('exploratory_exception',msg)
    raise


In [ ]:
# ===== R59. 额外年份确认（仅资格通过时） =====
R59_CONFIRM={'passed':False,'selected':None,'reason':'Exploratory Gate not passed'}
if R59_EXPLORATORY['passed'] and cfg.RUN_CONFIRMATION:
    try:
        qa=run_r59_phase(CONFIRMS,'额外确认')
        R59_CONFIRM=r59_gate('额外确认',CONFIRM_QA_YEARS)
        archive_r59('confirmation_finished')
    except Exception:
        msg=traceback.format_exc()
        (REP/'R59_异常回溯.txt').write_text(msg,encoding='utf-8')
        archive_r59('confirmation_exception',msg)
        raise
else:
    print('[R59 STOP] QA Gate failed or confirmation disabled. Save all negative evidence.')
    archive_r59('confirmation_skipped')


In [ ]:
# ===== R59. 汇总图、完整研究结论及自动ZIP =====
for phase in ('探索QA','额外确认'):
    f=DATA/f'R59_04_{phase}_模型汇总.csv'
    if not f.exists():continue
    means=pd.read_csv(f)
    palette={'Matched-LSTM':'#7895b2','R53-CapacityMatchedLSTM':'#4b6d95',
             'R53-ChannelConv-PCConcat':'#566f95',
             'R59-OwnBranch':'#a8a8a8','R59-CrossBranch':'#c77f4e','R59-HybridCoupled':'#c35337'}
    for zh,folder in ((True,FZH),(False,FEN)):
        old=plt.rcParams.copy()
        plt.rcParams['font.family']=[CJK_NAME] if zh else ['DejaVu Sans']
        fig,ax=plt.subplots(figsize=(8,4.6))
        for name in MODELS:
            s=means[means['模型']==name].sort_values('提前期_天')
            ax.plot(s['提前期_天'],s['RMSE_km'],marker='o',ms=5,lw=1.6,
                    color=palette[name],label=name)
        ax.set_xlabel('提前期（天）' if zh else 'Forecast lead (days)')
        ax.set_ylabel('横向路径误差 RMSE (km)' if zh else 'Cross-track RMSE (km)')
        ax.set_title('R5.9 跨分支残差校正对比' if zh else 'R5.9 cross-branch residual correction')
        ax.grid(alpha=.16);ax.legend(fontsize=8);fig.tight_layout()
        suffix='中文' if zh else 'English'
        fig.savefig(folder/f'R59_{phase}_预测误差_{suffix}.png',dpi=300,bbox_inches='tight')
        fig.savefig(folder/f'R59_{phase}_预测误差_{suffix}.pdf',bbox_inches='tight')
        plt.close(fig);plt.rcParams.update(old)
final={'research_version':'R59-cross-branch-residual-v1',
       'model_is_parent_LSTM_or_R53_hybrid_plus_cross_branch_low_rank_corrector':True,
       'exploratory':R59_EXPLORATORY,'confirmation':R59_CONFIRM,
       'ready_for_full_multiyear_experiment':bool(R59_EXPLORATORY['passed'] and R59_CONFIRM.get('passed',False)),
       'benchmark_honesty':'不能保证任何未见年份全面领先LSTM；旧开发QA多次暴露，不得称盲测；不从QA误差调alpha'}
(REP/'R59_最终交接.json').write_text(json.dumps(final,ensure_ascii=False,indent=2),encoding='utf-8')
archive_r59('completed_all')
print('[R59 RESULT]',json.dumps(final,ensure_ascii=False,indent=2))


## 科研解读限制

- 本脚本直接追求比强 LSTM 更低的横向路径误差，但**不会**事先保证或伪造改进幅度。内层验证校正失败会自动回退到父 LSTM，因此候选未必一定在 QA 优于父模型。
- 双分支父模型及较大 LSTM 保持与 R5.7 相同训练设置。校正器虽参数少，但使用额外训练更新和第二阶段验证选择，必须如实报告；正式论文中需提供相应训练预算说明。
- 2008、2016、2010、2018 已在历史研发中暴露，所有阳性结果只能作为研发资格证据。后续正式评估要对所有预先固定年份完整披露，不按结果挑选。
- 自动压缩包保留每个失败和回退记录，不包括完整权重；如要完整复现需另外备份 `模型权重/` 和 `教师预测缓存/`。